# Week 3 Helper Notebook 
Training SCARF: Contrastive Learning for Tabular Data
> WARNING: This code was generated as a skeleton for you all and may not work right out of the box. It was developed with assistance from Claude and not fully tested on the eICU dataset. Thus, if something doesn't work, or you see something missing, you should engineer it yourself. This is STARTER code and should not be treated as the final code!!

In this notebook you will train **SCARF**, which learns a vector representation $z$ using contrastive learning over tabular data. Unlike VAEs, SCARF has only an encoder and no decoder (so thus you can not directly reconstruct x from z!). The basic idea:
1. Take a patient row / data point `x`.
2. Make a **corrupted copy** `x~`: randomly pick some variables and replace their values with values taken from other rows.
3. Train an encoder so that `x` and `x~` get **similar embeddings** `z` and `z~`, while different rows (think of a different patient sample) get **different embeddings**.

To tell "my own corrupted copy" from "someone else's row", the encoder has to learn which combinations of values are typical, i.e. the structure of the table. This is the general objective of contrastive learning, which is often applied in image-based representation learning as well. 

**How is SCARF different from a VAE?** Here, we have no reconstruction option (bc we don't have a decoder) and no sampling (bc we aren't learning a defined distribution over `q(z\mid x)`).


**Files in this folder** (same preprocessing as the VAE notebook):

| File | Purpose |
|---|---|
| `encoders.py`, `preprocessing.py` | the same preprocessing as for the VAE (one-hot categoricals, encoded continuous variables, missing values) |
| `scarf_model.py` | corruption, the SCARF model, and the contrastive loss |

Runs on the **CPU**. *Reference implementation:* [clabrugere/pytorch-scarf](https://github.com/clabrugere/pytorch-scarf) (MIT license), simplified here.

In [ ]:
%load_ext autoreload
%autoreload 2
# This code helps with autoloading new files / modules. 

In [ ]:
import copy
import pickle as pk 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from proj34_representation.tabular_vae.scarf_model import SCARF, corrupt, nt_xent, variable_blocks
from proj34_representation.tabular_vae.encoders import *
from proj34_representation.tabular_vae.vae import TabularVAE, vae_loss
from proj34_representation.tabular_vae.preprocessing import TabularPreprocessor

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## Step 1: Load eICU data

In [ ]:
feature_sets = pk.load(open('../../data/feature_names.pkl', 'rb'))
patient_agg = pd.read_parquet('../../data/clean_dataset.parquet')

print("Dataset shape:", patient_agg.shape)
patient_agg.head()

In [ ]:
# For privacy, eICU records patients older than 89 using the string "> 89".
# This is how I chose to handle this, but you are free to change it! 
if 'age' in patient_agg.columns:
    patient_agg.loc[patient_agg.age == '> 89', 'age'] = 90
    patient_agg['age'] = patient_agg['age'].astype(float)

In [ ]:
[x for x in patient_agg.columns if 'mortality' in x]

In [ ]:
# Same as last week :) 
# TODO: it may be easier to add on your Y_cols for eval linear probing 
# here then dropping during SCARF training! 
X_cols = ['admissiondx_Diabetic ketoacidosis',
            'admissiondx_Seizures (primary-no structural brain disease)',
            'admissiondx_Rhythm disturbance (conduction defect)',
            'admissiondx_Rhythm disturbance (atrial, supraventricular)',
            'admissiondx_Respiratory - medical, other',
            'admissiondx_Renal failure, acute',
            'admissiondx_Pneumonia, bacterial',
            'admissiondx_Infarction, acute myocardial (MI)',
            'admissiondx_Hypertension, uncontrolled (for cerebrovascular accident-see Neurological System)',
            'admissiondx_Hemorrhage/hematoma, intracranial',
            'admissiondx_Head only trauma',
            'admissiondx_Emphysema/bronchitis',
            'admissiondx_Embolus, pulmonary',
            'admissiondx_Coma/change in level of consciousness (for hepatic see GI, for diabetic see Endocrine, if related to cardiac arrest, see CV)',
            'age','gender',
            'admissionheight',
            'admissionweight',
            'potassium',
            'sodium',
            'creatinine',
            'Hct',
            'BUN',
            'chloride',
            'Hgb',
            'glucose',
            'WBC x 1000',
            'RBC',
            'platelets x 1000',
            'calcium',
            'heartrate_bin1',
            'MCV',
            'MCHC',
            'sao2_bin1',
            'bicarbonate',
            'noninvasivemean_bin1',
            'noninvasivediastolic_bin1',
            'noninvasivesystolic_bin1',
            'RDW',
            'MCH',
            'respiration_bin1',
            'heartrate_bin2', 
            ]

num_cols = [x for x in X_cols if x in
    feature_sets['labs'].tolist()
    + feature_sets['vitals'].tolist()
    + ['age', 'admissionheight', 'admissionweight']
]

bin_cols = [x for x in X_cols if x in 
    feature_sets['icd10_before24h'].tolist()
    + feature_sets['admissiondx'].tolist()
]

cat_cols = [x for x in X_cols if x in [
    'hospital_region',
    'ethnicity',
    'gender',
    'hospital_numbedscategory',
    'hospitaldischargeyear',
    'hospitalid'
]]
cat_cols = bin_cols+cat_cols

df = patient_agg[X_cols]

## Step 1. Preprocess the data
Exactly the same steps as for the VAE: split first, fit the preprocessor on the **training** data only. Impute missing values.

In [ ]:
# TODO: use the same train / val / test split from Week2. ie. train_df should ALWAYS look the same!!
# You can generate test by applying train_test_split twice 
train_df, val_df, test_df = #train_test_split....

In [ ]:
# TODO: missingness imptuation strategy , as you explored Week2
imputed_train_df, imputed_val_df, imputed_test_df = ...

In [ ]:
preprocessor = TabularPreprocessor(cat_cols, num_cols)
x_train = preprocessor.fit_transform(imputed_train_df)
val_df = preprocessor.drop_unseen(imputed_val_df)
x_val = preprocessor.transform(imputed_val_df)
print(f"Each row has {preprocessor.input_dim_} numbers. Rows: {len(x_train)} train, {len(x_val)} validation")

## Step 2: X Corruption 
Investigate what corruption looks like. 
SCARF corrupts whole **variables**, not single columns. A categorical variable is a whole one-hot block, and replacing only part of it
would create an impossible row. `variable_blocks` tells the model where each variable starts and ends in the matrix.


You should be able to explain
what corruption on tabular data means in SCARF, and if you think these corruption
strategies are appropriate or not for tabular patient data. 

**Remember, 
a "corrupted" x is pushed to look the same in latent space as the original x
==> corruption should still be essentially the "same patient"!**

In [ ]:
blocks = variable_blocks(preprocessor)
print("Column range of each variable:", blocks)

batch = torch.from_numpy(x_train[:6])

# TODO: look into corrupt(). play around with this. 
corrupted = corrupt(batch, blocks, corruption_rate=0.3)

original_rows = preprocessor.inverse_transform(batch.numpy())
corrupted_rows = preprocessor.inverse_transform(corrupted.numpy())

# TODO: investigate original and corrupted rows. 

## Step 3: Let's train a simple SCARF model!

In [ ]:
# ----------------------- params you can play with -----------------------
EPOCHS = 100
BATCH_SIZE = 256          # the other rows in the batch are the "negatives", so batch size matters
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EMB_DIM = 16              # size of the embedding of each row
HIDDEN_DIM = 64
CORRUPTION_RATE = 0.3     # share of variables replaced in the corrupted copy (the reference repo uses 0.6, but
                          # our toy table has only 5 variables, so 0.3 works better; see exercise 1)
TEMPERATURE = 0.5         # lower = the loss focuses more on the hardest negatives (the reference repo uses 1.0)
VAL_FRACTION = 0.2
SEED = 42

USE_WANDB = False         # set to True to log to Weights & Biases (needs `pip install wandb` and `wandb login`)
WANDB_PROJECT = "tabular-scarf"
# --------------------------------------------------------------------------

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# A `DataLoader` hands the data to the model in small random **batches**.
train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train)), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(x_val)), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# TODO: read thru the scarf model 
model = SCARF(
    input_dim=preprocessor.input_dim_,
    blocks=blocks,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    corruption_rate=CORRUPTION_RATE,
)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(model)
print(f"Chance-level contrastive accuracy: {1 / (2 * BATCH_SIZE - 1):.4f}")

### Train

`run_epoch` makes one pass through a dataset. If you supply an optimizer it trains; without one it only evaluates.

**Validation detail:** the corruption of x is random, so the validation loss would be a bit different every epoch even if the model did not change.
We fix the random seed *only while validating* so the validation curve is comparable across epochs (`fork_rng` restores the training randomness afterwards).

In [ ]:
def run_epoch(model, loader, optimizer=None):
    """One pass over `loader`. Returns (average loss, average contrastive accuracy)."""
    training = optimizer is not None
    model.train(training)

    total_loss, total_accuracy, n_rows = 0.0, 0.0, 0

    with torch.set_grad_enabled(training):
        for (x,) in loader:
            z_anchor, z_corrupted = #TODO   forward pass (corrupts x inside)
            loss, accuracy = # TODO, hint: look into scarf_model

            if training:
                optimizer.zero_grad()      # 1) clear old gradients
                loss.backward()            # 2) compute new gradients
                optimizer.step()           # 3) update the weights

            total_loss += loss.item() * x.size(0)
            total_accuracy += accuracy * x.size(0)
            n_rows += x.size(0)

    return total_loss / n_rows, total_accuracy / n_rows

In [ ]:
if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, config={
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "emb_dim": EMB_DIM, "hidden_dim": HIDDEN_DIM, "corruption_rate": CORRUPTION_RATE,
        "temperature": TEMPERATURE,
    })

history = []
best_val_loss = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer)

    with torch.random.fork_rng(devices=[]):      # fixed corruption for validation (see text above)
        torch.manual_seed(0)
        val_loss, val_acc = run_epoch(model, val_loader)

    row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
           "train_acc": train_acc, "val_acc": val_acc}
    history.append(row)
    if USE_WANDB:
        wandb.log(row)

    note = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        note = "  <- best so far"

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:3d} | loss train {train_loss:6.3f} val {val_loss:6.3f}"
              f" | accuracy train {train_acc:5.3f} val {val_acc:5.3f}{note}")

model.load_state_dict(best_state)                # go back to the best epoch
history = pd.DataFrame(history)
if USE_WANDB:
    wandb.finish()

## Step 4: Understand the loss curves

* Loss going **down** and accuracy going **up** on both train and validation: the encoder is learning what belongs together.
* Train much better than validation: overfitting. Try a smaller network, more weight decay, or more corruption.
* A lower contrastive loss is **not** automatically a better embedding. (Ask yourself -- why not? Is there a degenerate trivial solution?)
* Is accuracy still far above chance?
* The validation loss can sit *below* the training loss. The training number is averaged while the model is still changing during the epoch, and validation uses a fixed corruption.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].plot(history["epoch"], history["train_loss"], label="train")
axes[0].plot(history["epoch"], history["val_loss"], label="validation")
axes[0].set_title("Contrastive loss (NT-Xent)")

axes[1].plot(history["epoch"], history["train_acc"], label="train")
axes[1].plot(history["epoch"], history["val_acc"], label="validation")
axes[1].axhline(1 / (2 * BATCH_SIZE - 1), color="gray", linestyle="--", label="chance")
axes[1].set_title("Contrastive accuracy")

for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

### Repeat training until val loss goes down. 

In [ ]:
# TODO: keep trying this, debugigng the code, etc. until the loss function decrease. 
# Scarf should be easier to train than VAEs though. 

## Exercise. Are the embeddings $z$ any good? Test linear probe

As in last week, we'll look at a common way to check a learned representation using a **linear probe**: if a simple linear model can predict an outcome Y from the latent vector z, then z has captured useful information about x.

3. Fit a linear model on the training rows only: **z → Y**. Use linear regression for a continuous Y, or logistic regression for a binary one.
4. Evaluate on the validation rows, using the **same train/validation split** as SCARF / VAE.
5. Compare with a linear model trained on **x → Y**. Which does better? Why might z be worse (or better)?

NEW: however that the dimension of z is often much less than the dimension of x --> adds a confounder of model size! So another "baseline comparison" is to look at a some z_random from a "random encoder" (in the linear setting, this is equivalent to a random projection onto some space) that is the same size as SCARF but not trained. Then, we can compare three probe performances : $X->Y$, $Z->Y$, $Z_{random}->Y$.

Often, if dim of X is small, then the raw point X is already very informative, so it is hard to beat. Embeddings pay off more on wide tables, or when you have few labels. What we can check here: does the SCARF embedding beat the *random* encoder, and does it get close to the raw row using far fewer numbers? The random-encoder score changes a lot from seed to seed, so change `SEED` and re-run before drawing conclusions.

In [ ]:
# TODO: compare validation R2 or AUPRC / AUROC across the 3 settings above. 

In [ ]:
# TODO (OPTIONAL but recommended): make a picture of the (val / test) embedding space using PCA 
# and color them by some variable X_i (ie. gender, mortality, age...). 
# Then compare to PCA using X.
# If SCARF works, Z's of the same "type of patient" should end up close to each other.

## Optional: Exercises

1. Try `CORRUPTION_RATE` = 0.1, 0.3, 0.6, 0.9 and re-run from section 4. What happens to the loss? To the probe score? Why would 0 corruption be useless?
3. Change `EMB_DIM` (2, 4, 16, 64). How many numbers does the embedding need?
